# Temperature Persona Hist Gradient Boosting Training

This notebook trains a scikit-learn `HistGradientBoostingClassifier` for temperature/HVAC persona prediction from `temperatureData.csv`. It saves app-facing artifacts to Google Drive under `AIModelsAndAlgorithms/TempreturePersona/`.

The visualization backend will use this model for the Temperature Persona card when `tempreture_persona_hgb_model.joblib` is present.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Keep these versions close to the app requirements and the other Colab exports.
!pip -q install scikit-learn==1.6.1 joblib==1.5.3 pandas==2.2.2 numpy==2.0.2 openpyxl==3.1.5


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
CANDIDATE_DATA_PATHS = [
    DRIVE_ROOT / 'VisualizationApp' / 'Data' / 'temperatureData.csv',
    DRIVE_ROOT / 'VisualizationApp' / 'temperatureData.csv',
    DRIVE_ROOT / 'Data' / 'temperatureData.csv',
    DRIVE_ROOT / 'temperatureData.csv',
]

DATA_CSV = next((p for p in CANDIDATE_DATA_PATHS if p.exists()), CANDIDATE_DATA_PATHS[0])
OUT_DIR = DRIVE_ROOT / 'VisualizationApp' / 'AIModelsAndAlgorithms' / 'TempreturePersona'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_CSV:', DATA_CSV)
print('DATA_CSV exists:', DATA_CSV.exists())
print('OUT_DIR:', OUT_DIR)

if not DATA_CSV.exists():
    raise FileNotFoundError(
        'Could not find temperatureData.csv in the default Drive locations. '
        'Set DATA_CSV manually to your file path, then rerun this cell.'
    )


DATA_CSV: /content/drive/MyDrive/VisualizationApp/Data/temperatureData.csv
DATA_CSV exists: True
OUT_DIR: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona


In [4]:
import json
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
MAX_ROWS = 500_000  # Increase gradually in Colab. Use None only with enough RAM.
MIN_CLASS_ROWS = 25

USECOLS = [
    'timestamp',
    'room_number',
    'floor',
    'facade',
    'room_type',
    'size_m2',
    'outside_temp',
    'room_temp',
    'setpoint',
    'ideal_temp',
    'hvac_mode',
    'ac_persona',
    'occupant_state',
    'pir_persona',
    'room_state',
    'pir_motion',
    'guest_id',
]

NUMERIC_FEATURES = [
    'floor_scaled',
    'size_scaled',
    'outside_temp_scaled',
    'room_temp_scaled',
    'setpoint_scaled',
    'ideal_temp_scaled',
    'temp_error_scaled',
    'comfort_error_scaled',
    'pir_motion',
    'has_guest',
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
]

CATEGORICAL_FEATURES = [
    'facade',
    'room_type',
    'hvac_mode',
    'occupant_state',
    'pir_persona',
    'room_state',
]

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = 'temperature_persona'


In [5]:
def clean_label(value) -> str:
    if pd.isna(value):
        return 'Unknown'
    text = str(value).strip()
    if text.lower() in {'', 'none', 'nan', 'null'}:
        return 'Unknown'
    return text


def cyclic(series: pd.Series, period: int) -> tuple[pd.Series, pd.Series]:
    radians = 2 * np.pi * series / period
    return np.sin(radians), np.cos(radians)


def scale_temp(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce').fillna(0).clip(-20, 50)
    return (values + 20) / 70


def build_features(max_rows: int | None = MAX_ROWS) -> pd.DataFrame:
    df = pd.read_csv(
        DATA_CSV,
        usecols=USECOLS,
        parse_dates=['timestamp'],
        nrows=max_rows,
    )
    print('Loaded rows:', len(df))
    print('Raw memory MB:', round(df.memory_usage(deep=True).sum() / 1024**2, 2))

    df = df.dropna(subset=['timestamp']).copy()
    df = df.sort_values(['room_number', 'timestamp']).reset_index(drop=True)

    for col in ['floor', 'size_m2', 'outside_temp', 'room_temp', 'setpoint', 'ideal_temp', 'pir_motion']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['guest_id'] = pd.to_numeric(df['guest_id'], errors='coerce')

    df[TARGET] = df['ac_persona'].map(clean_label)
    df = df[df[TARGET] != 'Unknown'].copy()

    class_counts = df[TARGET].value_counts()
    keep_classes = class_counts[class_counts >= MIN_CLASS_ROWS].index
    df = df[df[TARGET].isin(keep_classes)].copy()
    print('Rows after target cleanup:', len(df))
    print('Classes kept:')
    print(df[TARGET].value_counts())

    for col in CATEGORICAL_FEATURES:
        df[col] = df[col].map(clean_label)

    df['floor_scaled'] = df['floor'].clip(0, 30) / 30.0
    df['size_scaled'] = df['size_m2'].clip(0, 120) / 120.0
    df['outside_temp_scaled'] = scale_temp(df['outside_temp'])
    df['room_temp_scaled'] = scale_temp(df['room_temp'])
    df['setpoint_scaled'] = scale_temp(df['setpoint'])
    df['ideal_temp_scaled'] = scale_temp(df['ideal_temp'])
    df['temp_error_scaled'] = ((df['room_temp'] - df['setpoint']).clip(-20, 20) + 20) / 40.0
    df['comfort_error_scaled'] = ((df['room_temp'] - df['ideal_temp']).clip(-20, 20) + 20) / 40.0
    df['pir_motion'] = df['pir_motion'].clip(0, 1).astype('float32')
    df['has_guest'] = df['guest_id'].notna().astype('float32')
    df['hour_sin'], df['hour_cos'] = cyclic(df['timestamp'].dt.hour, 24)
    df['dow_sin'], df['dow_cos'] = cyclic(df['timestamp'].dt.dayofweek, 7)

    for col in NUMERIC_FEATURES:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('float32')

    return df.reset_index(drop=True)


In [6]:
df = build_features(MAX_ROWS)

if df.empty:
    raise ValueError('No labeled temperature persona rows found after cleanup. Check ac_persona values.')

split_time = df['timestamp'].quantile(0.80)
train_df = df[df['timestamp'] <= split_time].copy()
test_df = df[df['timestamp'] > split_time].copy()

# If the time split removes a rare class from test, fall back to a stratified random split.
missing_test_classes = set(train_df[TARGET].unique()) - set(test_df[TARGET].unique())
if len(test_df) == 0 or missing_test_classes:
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(
        df,
        test_size=0.20,
        random_state=SEED,
        stratify=df[TARGET],
    )

x_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET]
x_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET]

print('Train rows:', len(train_df))
print('Test rows:', len(test_df))
print('Train classes:', sorted(y_train.unique()))


Loaded rows: 500000
Raw memory MB: 199.76
Rows after target cleanup: 33595
Classes kept:
temperature_persona
Reactive           17041
EnergySaver        12233
AlwaysOnComfort     3196
Preconditioning      780
Housekeeping         345
Name: count, dtype: int64
Train rows: 26883
Test rows: 6712
Train classes: ['AlwaysOnComfort', 'EnergySaver', 'Housekeeping', 'Preconditioning', 'Reactive']


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            NUMERIC_FEATURES,
        ),
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
            ]),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder='drop',
)

model = Pipeline([
    ('preprocessor', preprocessor),
    (
        'classifier',
        HistGradientBoostingClassifier(
            max_iter=250,
            learning_rate=0.06,
            max_leaf_nodes=31,
            l2_regularization=0.05,
            early_stopping=True,
            random_state=SEED,
        ),
    ),
])

model.fit(x_train, y_train)
pred = model.predict(x_test)
proba = model.predict_proba(x_test) if hasattr(model, 'predict_proba') else None
classes = [str(c) for c in model.classes_]

metrics = {
    'accuracy': float(accuracy_score(y_test, pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
}
print(metrics)
print(classification_report(y_test, pred, digits=4))


{'accuracy': 0.9000297973778307, 'balanced_accuracy': 0.888956838905775}
                 precision    recall  f1-score   support

AlwaysOnComfort     0.6429    0.5246    0.5777       875
    EnergySaver     1.0000    1.0000    1.0000      2446
   Housekeeping     1.0000    1.0000    1.0000        38
Preconditioning     1.0000    1.0000    1.0000       157
       Reactive     0.8761    0.9202    0.8976      3196

       accuracy                         0.9000      6712
      macro avg     0.9038    0.8890    0.8951      6712
   weighted avg     0.8944    0.9000    0.8962      6712



In [8]:
model_file = OUT_DIR / 'tempreture_persona_hgb_model.joblib'
metadata_file = OUT_DIR / 'tempreture_persona_hgb_metadata.json'
report_file = OUT_DIR / 'tempreture_persona_hgb_report.txt'
confusion_file = OUT_DIR / 'tempreture_persona_hgb_confusion_matrix.csv'
predictions_file = OUT_DIR / 'tempreture_persona_hgb_sample_predictions.csv'
zip_file = OUT_DIR / 'tempreture_persona_hgb_outputs.zip'

report_text = classification_report(y_test, pred, digits=4)
cm = pd.DataFrame(confusion_matrix(y_test, pred, labels=classes), index=classes, columns=classes)
cm.to_csv(confusion_file)

sample = test_df[['timestamp', 'room_number', 'room_temp', 'setpoint', 'ideal_temp', 'hvac_mode', 'room_state', 'ac_persona']].copy()
sample['model_prediction'] = pred
if proba is not None:
    sample['model_confidence'] = proba.max(axis=1)
sample.head(5000).to_csv(predictions_file, index=False)

metadata = {
    'model_type': 'hist_gradient_boosting_classifier',
    'task': 'tempreture_persona',
    'target': TARGET,
    'classes': classes,
    'feature_columns': FEATURE_COLUMNS,
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'seed': SEED,
    'max_rows': MAX_ROWS,
    'min_class_rows': MIN_CLASS_ROWS,
    'output_dir': str(OUT_DIR),
    'metrics': metrics,
}

bundle = {
    'model': model,
    'feature_columns': FEATURE_COLUMNS,
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'classes': classes,
    'metadata': metadata,
}
joblib.dump(bundle, model_file)
metadata_file.write_text(json.dumps(metadata, indent=2))
report_file.write_text(report_text)

with zipfile.ZipFile(zip_file, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [model_file, metadata_file, report_file, confusion_file, predictions_file]:
        zf.write(path, arcname=path.name)

print('Saved files:')
for path in [model_file, metadata_file, report_file, confusion_file, predictions_file, zip_file]:
    print(path)


Saved files:
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/tempreture_persona_hgb_model.joblib
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/tempreture_persona_hgb_metadata.json
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/tempreture_persona_hgb_report.txt
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/tempreture_persona_hgb_confusion_matrix.csv
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/tempreture_persona_hgb_sample_predictions.csv
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/tempreture_persona_hgb_outputs.zip


In [9]:
# Quick sanity check: reload the saved artifact and predict one row.
loaded = joblib.load(model_file)
row = x_test.head(1)
print('Reloaded keys:', loaded.keys())
print('Prediction:', loaded['model'].predict(row)[0])
print('Classes:', loaded['classes'])


Reloaded keys: dict_keys(['model', 'feature_columns', 'numeric_features', 'categorical_features', 'classes', 'metadata'])
Prediction: Preconditioning
Classes: ['AlwaysOnComfort', 'EnergySaver', 'Housekeeping', 'Preconditioning', 'Reactive']
